# 33. Does the literature's plasticity metric agree with direct measurement here?

Matas found that the usual plasticity metrics disagree with direct measurement in one
setting. Three of his four cells agree. The fourth, continual learning with adapters, does
not: the adapter learns new tasks to above 99 percent while the metrics report a large loss.
Giving the head its own faster learning rate moved effective rank from 0.6 to over 0.94
without changing what the adapter could learn, which says the metric was tracking the head
rather than the learning.

His explanation blames the head's learning rate. Our architecture does not have one.
`uncle/trainer.py` builds a single Adam over all trainable parameters at one rate, and the
classifier is produced by the generator rather than trained on its own. There is also no
frozen backbone.

So this is the control for his result, and both answers are useful. If the metric drops here
too, the head learning rate cannot be the whole story and his claim widens beyond adapters.
If it does not drop, his explanation is confirmed from an architecture that lacks the cause.

**This notebook runs nothing.** Notebook 29 already recorded effective rank beside the
accuracy at every probe point. This downloads that file and lays the two side by side.
Seconds, no GPU, no images.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util, statistics
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/path-rank")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
if importlib.util.find_spec("boto3") is None:
    if not IN_COLAB:
        raise ImportError("Install boto3 first.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3"], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.probe import best_of_grid

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)

## 2. Settings

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"

# Notebook 29 wrote every finished probe run here, one JSON object per line.
RESUME_KEY = f"uncle/plasticity_probe/{STUDY_ID}/{ORDER}/partial.jsonl"

# The full arm is the one to read: generator and task code both train, which is the
# closest thing here to the adapter study's own probe. E2 is the natural-task endpoint,
# the one that carries both a test accuracy and an effective rank.
ARM = "full"
ENDPOINT = "E2"

CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Reading", RESUME_KEY)

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
if store.info(RESUME_KEY) is None:
    raise RuntimeError(f"No saved probe runs at {RESUME_KEY}. Run notebook 29 first.")
saved = store.download(RESUME_KEY, CACHE / "cache")

runs = []
for line in saved.read_text(encoding="utf-8").splitlines():
    try:
        runs.append(json.loads(line))
    except json.JSONDecodeError:
        print("Skipped one unreadable line at the end of the saved file.")
print(f"Read {len(runs)} probe runs")

## 4. Pick one run per measurement

Notebook 29 ran a grid of learning rates and kept the best. We repeat that choice here with
the same function the notebook used, so these numbers are the ones its own tables report and
not a different summary of the same file.

In [ ]:
wanted = [r for r in runs if r.get("arm") == ARM and r.get("endpoint") == ENDPOINT]
print(f"{len(wanted)} runs in the {ARM} arm, endpoint {ENDPOINT}")
if not wanted:
    raise RuntimeError("Nothing to read. Check ARM and ENDPOINT against notebook 29.")

# Group by everything except the learning rate, then let best_of_grid choose the rate.
groups = {}
for run in wanted:
    key = (run["seed_label"], run["point"], run["task"], run["repetition"])
    groups.setdefault(key, []).append(run)
picked = {key: best_of_grid(group, endpoint=ENDPOINT) for key, group in groups.items()}
print(f"{len(picked)} measurements after choosing a rate for each")

POINTS = sorted({key[1] for key in picked})
SEEDS = sorted({key[0] for key in picked})
print("Probe points:", POINTS)
print("Seeds:", SEEDS)

## 5. The two measurements side by side

For each seed and probe point, the mean over probe tasks and repetitions. Accuracy is the
direct measurement: can the model learn a task it has never seen. Effective rank is the
literature's indirect measurement, computed from the features of the same runs.

In [ ]:
def mean_of(key_filter, field):
    values = [run[field] for key, run in picked.items()
              if key_filter(key) and run.get(field) is not None]
    return (statistics.mean(values), statistics.stdev(values) if len(values) > 1 else float("nan"),
            len(values))

print(f"{'seed':>5} {'point':>7} {'accuracy %':>12} {'sd':>7} "
      f"{'eff rank':>10} {'sd':>7} {'runs':>6}")
table = {}
for seed in SEEDS:
    for point in POINTS:
        pick = lambda key, s=seed, p=point: key[0] == s and key[1] == p
        accuracy, accuracy_sd, count = mean_of(pick, "test_accuracy")
        rank, rank_sd, _ = mean_of(pick, "effective_rank")
        table[(seed, point)] = {"accuracy": accuracy, "effective_rank": rank, "runs": count}
        print(f"{str(seed):>5} {str(point):>7} {accuracy:>12.2f} {accuracy_sd:>7.2f} "
              f"{rank:>10.2f} {rank_sd:>7.2f} {count:>6}")

## 6. Do the two agree?

The question is whether they move in the same direction between the first and last probe
point. Agreement means both say the model is fine, or both say it is worse. Disagreement is
the anomaly Matas found.

In [ ]:
first, last = POINTS[0], POINTS[-1]
print(f"From point {first} to point {last}\n")
print(f"{'seed':>5} {'accuracy change':>16} {'rank change':>13} {'rank change %':>14} {'agree?':>8}")
disagreements = 0
for seed in SEEDS:
    start, end = table[(seed, first)], table[(seed, last)]
    accuracy_change = end["accuracy"] - start["accuracy"]
    rank_change = end["effective_rank"] - start["effective_rank"]
    rank_percent = 100.0 * rank_change / start["effective_rank"] if start["effective_rank"] else float("nan")
    # Agreement means neither says the model got worse, or both do. A rank that falls
    # while accuracy rises is the disagreement the adapter study reports.
    agree = not (accuracy_change > 0 and rank_change < 0) and not (accuracy_change < 0 and rank_change > 0)
    disagreements += 0 if agree else 1
    print(f"{str(seed):>5} {accuracy_change:>16.2f} {rank_change:>13.2f} "
          f"{rank_percent:>13.1f}% {('yes' if agree else 'NO'):>8}")

print()
if disagreements == 0:
    print("The two measurements agree on every seed.")
    print("In this architecture the metric does not report a loss that direct measurement denies.")
    print("That is the result the head learning rate explanation predicts, because there is")
    print("no separate head learning rate here for the metric to track.")
else:
    print(f"The two measurements disagree on {disagreements} of {len(SEEDS)} seeds.")
    print("The anomaly appears in an architecture with no head learning rate and no frozen")
    print("backbone, so the head learning rate cannot be the whole explanation.")
print()
print("One line for the write-up:")
for seed in SEEDS:
    start, end = table[(seed, first)], table[(seed, last)]
    print(f"  seed {seed}: accuracy {start['accuracy']:.1f} -> {end['accuracy']:.1f}, "
          f"effective rank {start['effective_rank']:.1f} -> {end['effective_rank']:.1f}")

## What this is, and what it is not

**What it is.** At each probe point a copy of the model is trained on a task it has never
seen, scored, and discarded. Accuracy says how well it learned. Effective rank is computed
from the features of that same copy. So the two columns describe one model, measured two
ways, which is what makes a disagreement meaningful.

**The difference from Matas's setup, which he should be told.** He measures effective rank on
the continually trained model itself. These numbers come from the model after it has been
adapted to the probe task for 250 steps. Both are measurements of a continually trained
system, but they are not the same measurement, and the comparison should say so rather than
imply the numbers are interchangeable.

**Effective rank here is an absolute count**, out of 768 feature dimensions. Matas reports
values between 0 and 1, which look like a normalised version. Confirm which before putting
the two in one table, or 0.94 will be compared with 300 and the table will look broken.

**Three seeds.** The spread column is there so nobody reads a small change as a result.

**What would make this stronger.** Effective rank measured along the stream rather than only
at its ends, which needs the per-task checkpoints that batch 5 of the TODO unblocks.